# Lab 7 – RAG

In [3]:

!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install faiss-cpu pymupdf sentence-transformers tqdm -q

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (7,495 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently 

In [4]:

import subprocess, time

subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

!ollama pull llama3.2
print('Ollama gotowa!')


Ollama gotowa!


In [5]:

import os
import json
import faiss
import pymupdf
import numpy as np
import requests
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
print('Embedder załadowany!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder załadowany!


In [6]:

index = faiss.IndexFlatL2(embedder.get_sentence_embedding_dimension())

metadata = []

print('Number of chunks: ', index.ntotal)  # 0

Number of chunks:  0


/tmp/ipykernel_3102/1384208494.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  index = faiss.IndexFlatL2(embedder.get_sentence_embedding_dimension())


In [7]:

class Utils:
    def __init__(self, embedding_model, index, metadata, chunk_size=512, ollama_model='llama3.2'):
        self.embedding_model = embedding_model
        self.index = index
        self.metadata = metadata
        self.chunk_size = chunk_size
        self.ollama_model = ollama_model

    def extract_text_from_pdf(self, pdf_path):
        """Wyciąga tekst z PDF. Zwraca listę krotek (numer_strony, tekst)."""
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)

            text.append((page_num, str(page.get_text()).replace('\n', ' ')))
        return text

    def chunk_text(self, text):
        """Dzieli tekst na fragmenty o długości chunk_size znaków."""
        chunks = []
        for page_num, page_text in text:
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc='vec_db/'):
        os.makedirs(db_loc, exist_ok=True)
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc='Adding chunks to FAISS')):

            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)

            self.index.add(np.array([embeddings]))

            self.metadata.append({
                'filename': filename,
                'page_number': page_number,
                'chunk_num': chunk_num,
                'chunk': chunk
            })

        faiss.write_index(self.index, db_loc + 'vector_database.index')
        with open(db_loc + 'metadata.json', 'w') as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        """Przetwarza plik PDF i dodaje chunki do FAISS."""
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f'Nieobsługiwany format: {os.path.splitext(file_path)[1]}')
            return 0
        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)

    def ask_ollama(self, prompt, system_prompt):
        """Wysyła zapytanie do lokalnego serwera Ollama i zwraca odpowiedź."""
        response = requests.post(
            'http://localhost:11434/api/chat',
            json={
                'model': self.ollama_model,
                'messages': [
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': prompt}
                ],
                'stream': False
            }
        )
        return response.json()['message']['content']

    def answer_question(self, prompt_template='', query='', k=5):

        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)

        D, I = self.index.search(np.array([question_embedding]), k)
        chunks = [self.metadata[i] for i in I[0]]

        context = ''
        for i, chunk in enumerate(chunks):
            context += f'{i+1}. {chunk["chunk"]}\n'

        prompt = prompt_template.format(context=context, query=query)

        system_prompt = (
            'Be helpful, straight to the point. '
            'Use only context. Do not hallucinate. '
            'If context does not contain the answer, say so in Polish.'
        )

        answer = self.ask_ollama(prompt, system_prompt)
        return answer, chunks

In [8]:

utils = Utils(
    embedding_model=embedder,
    index=index,
    metadata=metadata,
    chunk_size=512,
    ollama_model='llama3.2'
)

In [9]:

import os
from google.colab import files

os.makedirs('knowledge', exist_ok=True)
os.makedirs('vec_db', exist_ok=True)

print('Pliki PDF:')
uploaded = files.upload()

for fname in uploaded:
    with open(f'knowledge/{fname}', 'wb') as f:
        f.write(uploaded[fname])
    print(f'Zapisano: knowledge/{fname}')

Wgraj pliki PDF z moodle:


Saving Astrochemistry_of_dust.pdf to Astrochemistry_of_dust.pdf
Zapisano: knowledge/Astrochemistry_of_dust.pdf


In [10]:

knowledge_dir = 'knowledge/'
for file in os.listdir(knowledge_dir):
    if file.endswith('.pdf'):
        n = utils.process_file(knowledge_dir + file)
        print(f'{file}: {n} chunków')

print('\nNumber of chunks: ', index.ntotal)

Adding chunks to FAISS: 100%|██████████| 275/275 [02:47<00:00,  1.64it/s]

Astrochemistry_of_dust.pdf: 275 chunków

Number of chunks:  275


In [11]:

questions = [
    'Dlaczego astronomia kosmiczna jest ważna dla współczesnej nauki?',
    'Jak atmosfera Ziemi wpływa na obserwacje astronomiczne?',
    'Jakie rodzaje promieniowania elektromagnetycznego są wykorzystywane w astronomii?',
    'Czym różni się teleskop optyczny od radioteleskopu?',
    'Jakie informacje o obiektach kosmicznych można uzyskać dzięki analizie widma?',
    'Jak astronomowie wykorzystują podczerwień do badania kosmosu?',
    'Czym jest interferometria w astronomii?',
    'Jakie odkrycia umożliwiły obserwacje w zakresie fal radiowych?',
    'Czym są egzoplanety?',
    'Jakie są główne metody wykrywania egzoplanet?',
    'Dlaczego wykrywanie małych egzoplanet jest trudniejsze niż dużych?',
    'Czym jest strefa zamieszkiwalna wokół gwiazdy?',
    'Jak astronomowie badają atmosfery egzoplanet?',
    'Jakie cechy planety mogą wskazywać na możliwość istnienia życia?',
    'Jakie typy egzoplanet odkryto do tej pory?',
    'Czym jest astroML?',
    'Jak machine learning jest wykorzystywany w astronomii?',
    'Jakie typy danych astronomicznych analizuje się za pomocą ML?',
    'Dlaczego astronomia generuje duże ilości danych?',
    'Czym zajmuje się astrochemia?',
    'Jak powstają cząsteczki w przestrzeni międzygwiazdowej?',
    'Jaką rolę odgrywa pył kosmiczny w formowaniu gwiazd i planet?',
    'Jakie związki organiczne odkryto w obłokach molekularnych?',
    'Jakie znaczenie ma astrochemia dla badań nad pochodzeniem życia?',
]

In [12]:

prompt_template = """Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia).
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
    prompt_template=prompt_template,
    query=random_query,
    k=5
)

display(Markdown(f'**Pytanie:** {random_query}'))
display(Markdown(f'**Odpowiedź:**\n\n{response}'))
display(Markdown('---\n**Źródła:**'))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + '...'
    display(Markdown(
        f'**[{i+1}]** `{chunk["filename"]}` — strona {chunk["page_number"] + 1}\n\n'
        f'> {excerpt}'
    ))

**Pytanie:** Dlaczego astronomia generuje duże ilości danych?

**Odpowiedź:**

Astronomia generuje duże ilości danych z powodu postępującego rozwoju dużych teleskopów wyposażonych w niskoempiryczne detektorów, które pozwalają na badanie chmur gazu w różnych długościach fal i zakresach elektromagnetycznymu, każdy z których opowiada inny koniec historii.

---
**Źródła:**

**[1]** `Astrochemistry_of_dust.pdf` — strona 7

> ange from a few to tens of K, and are thus readily excited at typical dense cloud conditions. The advantage of mm observations is high sensitivity to low abundance molecules (down to 10−11 with respec...

**[2]** `Astrochemistry_of_dust.pdf` — strona 7

> tromagnetic spectrum, each of which tells a different part of the story. In the last decade astrochemistry has been very fortunate to have had access to a number of new powerful telescopes with both i...

**[3]** `Astrochemistry_of_dust.pdf` — strona 27

> in the cold interstellar clouds prior to collapse as in § 7.2. Following the water trail from dense cloud cores through collapsing envelopes to planet-forming disks and exoplanets is a major goal of m...

**[4]** `Astrochemistry_of_dust.pdf` — strona 2

> clouds, with 1000 times more hydrogen than any other chemically interesting el- ement. The detection of nearly 180 different species over the past 45 years (not counting isotopologs) demonstrates the...

**[5]** `Astrochemistry_of_dust.pdf` — strona 25

> young stars are the birthplaces of planets and are therefore partic- ularly important targets for astrochemistry. However, disks are at least a factor of 1000 smaller than the clouds in which they are...

In [13]:

moje_pytanie = 'Czym jest astroML?'

response, chunks = utils.answer_question(
    prompt_template=prompt_template,
    query=moje_pytanie,
    k=5
)

display(Markdown(f'**Pytanie:** {moje_pytanie}'))
display(Markdown(f'**Odpowiedź:**\n\n{response}'))
display(Markdown('---\n**Źródła:**'))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + '...'
    display(Markdown(
        f'**[{i+1}]** `{chunk["filename"]}` — strona {chunk["page_number"] + 1}\n\n'
        f'> {excerpt}'
    ))

**Pytanie:** Czym jest astroML?

**Odpowiedź:**

Nie ma informacji w dostępnym kontekście o definicji lub pojęci astroML.

---
**Źródła:**

**[1]** `Astrochemistry_of_dust.pdf` — strona 2

> tmospheres of giant exo-planets6,7. Astrochemistry, also known as molecular astrophysics, is ‘the study of the formation, destruction and excitation of molecules in astronomical environments and their...

**[2]** `Astrochemistry_of_dust.pdf` — strona 1

> s, with links to papers presented in this volume. In spite of many lingering uncertainties, the future of astrochemistry is bright: new observational facilities promise major advances in our understan...

**[3]** `Astrochemistry_of_dust.pdf` — strona 1

> iggered new insight into chemistry, illustrating how astron- omy and chemistry can enhance each other. Much of the chemistry in star- and planet-forming regions is now thought to be driven by gas-grai...

**[4]** `Astrochemistry_of_dust.pdf` — strona 2

> new chemical physics through studies of different classes of molecules and reactions that are normally not con- sidered on Earth. Indeed, astrochemistry is a ‘blending of astronomy and chem- istry in...

**[5]** `Astrochemistry_of_dust.pdf` — strona 1

> lso important because they are the nurseries of new generations of stars like our Sun and planets like Jupiter or Earth. This combi- nation makes astrochemistry such a fascinating research ﬁeld, for b...